# question_layout_v4-palme-all — Colab GPU eğitimi

Soru-özel DocLayout-YOLO modelini (4 sınıf: `question_anchor`, `question_body`,
`shared_stimulus`, `solution_or_answer`) DocStructBench ağırlığından transfer ile eğitir.

**Ön hazırlık (Mac'te, bir kez):**
1. Palme etiketlerini ve temiz v4 veri setini üret.
2. Paketle: `.venv/bin/python soru_model_colab_paketle.py --veri "$HOME/Library/Application Support/soru-kesici/egitim/ql_v4_palme_all" --cikti ~/Desktop/ql_v4_palme_all_colab.zip --arsiv-koku ql_v4_palme_all`
3. ZIP'i Google Drive'a yükle: **MyDrive/soru-kesici/ql_v4_palme_all_colab.zip**

**Burada:** Çalışma zamanı → Çalışma zamanı türünü değiştir → **GPU**
(T4 yeterli; L4/A100 belirgin hızlı). Eğitim çıktıları Drive'a yazılır.
Güvenlik için eski/pickle checkpoint'ten otomatik devam kapalıdır; oturum
koparsa yeni bir `RUN_ID` ile doğrulanmış başlangıç ağırlığından yeniden başlayın.


In [ ]:
!nvidia-smi


In [ ]:
# ------------------------- AYARLAR -------------------------
ZIP_YOLU = "/content/drive/MyDrive/soru-kesici/ql_v4_palme_all_colab.zip"
ZIP_SHA256 = "272c89585a2afb1af142b6b23f511b512e925c666096689a50893c6ce3f2728b"
PARCA_KALIBI = "/content/drive/MyDrive/soru-kesici/parcalar/ql_v4_palme_all_colab.zip.*"
ARSIV_KOKU = "ql_v4_palme_all"
RUN_ID = "palme-all-v4-20260731-r1"
CIKTI_KOKU = "/content/drive/MyDrive/soru-kesici/ql_v4_runs"
CIKTI_DIZIN = CIKTI_KOKU + "/" + RUN_ID
RESMI_BASE_SHA256 = "9a2ee0220fe3d9ad31b47e1d9f1282f46959a54e4618fce9cffcc9715b8286e2"
RESMI_BASE_BOYUT = 40709302
RESMI_BASE_REPO = "juliozhao/DocLayout-YOLO-DocStructBench"
RESMI_BASE_REVIZYON = "8c3299a30b8ff29a1503c4431b035b93220f7b11"
EPOCHS = 60
IMGSZ = 1280
PATIENCE = 12
RESUME = False   # güvenlik politikası: bu defterde True kabul edilmez


In [ ]:
import glob, hashlib, os, pathlib, shutil, stat, tempfile, zipfile
from google.colab import drive
drive.mount("/content/drive")

def dosya_sha256(yol):
    h = hashlib.sha256()
    with open(yol, "rb") as f:
        for parca in iter(lambda: f.read(1 << 20), b""):
            h.update(parca)
    return h.hexdigest()

def acik_dosya_sha256(dosya):
    konum = dosya.tell()
    dosya.seek(0)
    h = hashlib.sha256()
    for parca in iter(lambda: dosya.read(1 << 20), b""):
        h.update(parca)
    dosya.seek(konum)
    return h.hexdigest()

assert ZIP_SHA256 != "PAKETLENINCE_GUNCELLE", "ZIP_SHA256 ayarlanmamış"
kaynaklar = [ZIP_YOLU] if os.path.isfile(ZIP_YOLU) else []
if not kaynaklar:
        # Tek ZIP yoksa parçalardan birleştir (soru_model_colab_paketle.py
        # --parca-boyut çıktısı; tarayıcıdan parça parça yüklenir).
    kaynaklar = sorted(glob.glob(PARCA_KALIBI))
    assert kaynaklar, f"Ne ZIP ne parça bulundu: {ZIP_YOLU} / {PARCA_KALIBI}"
    print(f"{len(kaynaklar)} parça birleştiriliyor…")
VERI_KOKU = "/content/soru-kesici-datasets"
ACMA_KOKU = os.path.join(VERI_KOKU, ZIP_SHA256)
VERI = os.path.join(ACMA_KOKU, ARSIV_KOKU)
ISARET = os.path.join(ACMA_KOKU, ".archive_sha256")
os.makedirs(VERI_KOKU, exist_ok=True)
# Drive yolu değiştirilebilir: kaynak(lar)ı birer kez açıp aynı yerel staged
# FD'ye kopyala+hashle; güvenli çıkarma ve son hash boyunca bu FD'den ayrılma.
stage_fd, stage_yolu = tempfile.mkstemp(prefix=".ql-v4-stage-", suffix=".zip", dir=VERI_KOKU)
gecici = None
try:
    with os.fdopen(stage_fd, "w+b") as staged:
        kopya_sha = hashlib.sha256()
        for kaynak in kaynaklar:
            with open(kaynak, "rb") as k:
                for parca in iter(lambda: k.read(1 << 20), b""):
                    staged.write(parca)
                    kopya_sha.update(parca)
        staged.flush()
        os.fsync(staged.fileno())
        assert kopya_sha.hexdigest() == ZIP_SHA256, (kopya_sha.hexdigest(), ZIP_SHA256)
        os.fchmod(staged.fileno(), 0o400)
        assert acik_dosya_sha256(staged) == ZIP_SHA256
        print("Yerel staged ZIP SHA-256 doğrulandı:", ZIP_SHA256)
        # Her koşuda doğrulanmış ZIP'ten boş bir dizin üret; yazılabilir eski
        # çıkarma ağacına yalnız işaret dosyasına bakarak güvenme.
        gecici = tempfile.mkdtemp(prefix=".ql-v4-aciliyor-", dir=VERI_KOKU)
        staged.seek(0)
        with zipfile.ZipFile(staged) as z:
            uyeler = z.infolist()
            assert len(uyeler) <= 10000, "beklenmeyen ZIP üye sayısı"
            assert sum(u.file_size for u in uyeler) <= 6 * (1 << 30), "ZIP fazla büyük"
            gorulen = set()
            for u in uyeler:
                assert not (u.flag_bits & 1), "şifreli ZIP üyesi"
                ham = u.filename
                assert "\\" not in ham and "\x00" not in ham
                posix = pathlib.PurePosixPath(ham)
                parcalar = posix.parts
                assert parcalar and not posix.is_absolute()
                assert not any(p in ("", ".", "..") for p in parcalar), ham
                assert parcalar[0] == ARSIV_KOKU, ham
                kanonik = "/".join(parcalar) + ("/" if u.is_dir() else "")
                assert ham == kanonik, f"kanonik olmayan ZIP üyesi: {ham}"
                kimlik = kanonik.casefold()
                assert kimlik not in gorulen, "yinelenen ZIP üyesi"
                gorulen.add(kimlik)
                kip = (u.external_attr >> 16) & 0xFFFF
                assert not stat.S_ISLNK(kip), ham
                tip = stat.S_IFMT(kip)
                assert tip in (0, stat.S_IFREG, stat.S_IFDIR), ham
            z.extractall(gecici)
        assert acik_dosya_sha256(staged) == ZIP_SHA256, "staged ZIP çıkarma sırasında değişti"
        with open(os.path.join(gecici, ".archive_sha256"), "w", encoding="ascii") as f:
            f.write(ZIP_SHA256 + "\n")
        if os.path.isdir(ACMA_KOKU):
            shutil.rmtree(ACMA_KOKU)
        os.replace(gecici, ACMA_KOKU)
        gecici = None
finally:
    if gecici is not None:
        shutil.rmtree(gecici, ignore_errors=True)
    try:
        os.unlink(stage_yolu)
    except OSError:
        pass
assert os.path.isfile(os.path.join(VERI, "manifest.json")), \
    "manifest.json yok — ARSIV_KOKU ile ZIP kökü uyuşmuyor"
print("Veri seti hazır:", VERI)


In [ ]:
# Colab'ın hazır torch/CUDA kurulumuna dokunmamak için --no-deps; eksik kalan
# küçük bağımlılıklar ayrıca kurulur. Yerel .venv-ml ile aynı sürüm: 0.0.4.
%pip install -q --no-deps doclayout-yolo==0.0.4
%pip install -q "huggingface-hub>=0.23,<2" dill py-cpuinfo thop


In [ ]:
import json
man = json.load(open(os.path.join(VERI, "manifest.json"), encoding="utf-8"))
base_bilgi = man.get("base_model") or {}
base_guven = base_bilgi.get("trust_root") or {}
assert base_bilgi.get("sha256") == RESMI_BASE_SHA256
assert base_bilgi.get("size") == RESMI_BASE_BOYUT
assert base_guven.get("kind") == "pinned-official-huggingface-revision"
assert base_guven.get("repository") == RESMI_BASE_REPO
assert base_guven.get("revision") == RESMI_BASE_REVIZYON
base_rel = base_bilgi.get("path", "")
assert base_rel.startswith("base/") and ".." not in pathlib.PurePosixPath(base_rel).parts
BASLANGIC = os.path.realpath(os.path.join(VERI, *base_rel.split("/")))
assert os.path.commonpath((os.path.realpath(VERI), BASLANGIC)) == os.path.realpath(VERI)
assert os.path.isfile(BASLANGIC), "Paketlenmiş başlangıç ağırlığı yok"
assert os.path.getsize(BASLANGIC) == RESMI_BASE_BOYUT
assert dosya_sha256(BASLANGIC) == RESMI_BASE_SHA256
print("Başlangıç ağırlığı SHA-256 doğrulandı:", base_bilgi["sha256"])


In [ ]:
import importlib.metadata as importlib_metadata
import numpy as np
import torch

# numpy 2.x: np.trapz kaldırıldı; doclayout_yolo metrik hesabı eski adı çağırıyor.
if not hasattr(np, "trapz") and hasattr(np, "trapezoid"):
    np.trapz = np.trapezoid

SINIFLAR = ("question_anchor", "question_body",
            "shared_stimulus", "solution_or_answer")
assert tuple(man["classes"]) == SINIFLAR, "sınıf sırası sözleşmeyle uyuşmuyor"
assert man.get("package_schema_version") == "soru-kesici.question-layout-colab-package/v1"
assert man.get("training_package_fingerprint_version") == "soru-kesici.question-layout-colab-fingerprint/v3"
assert man.get("archive_root") == ARSIV_KOKU
print("huggingface-hub:", importlib_metadata.version("huggingface-hub"))

def veri_altinda(goreli, alan):
    assert isinstance(goreli, str) and goreli and "\\" not in goreli, alan
    posix = pathlib.PurePosixPath(goreli)
    assert not posix.is_absolute() and posix.as_posix() == goreli, alan
    assert not any(p in ("", ".", "..") for p in posix.parts), alan
    sonuc = os.path.realpath(os.path.join(VERI, *posix.parts))
    assert os.path.commonpath((os.path.realpath(VERI), sonuc)) == os.path.realpath(VERI), alan
    return sonuc

def doclayout_etiket_yolu(gorsel):
    tam = "/" + gorsel
    assert "/images/" in tam, gorsel
    kok, kuyruk = tam.rsplit("/images/", 1)
    return ((kok + "/labels/" + kuyruk).rsplit(".", 1)[0] + ".txt").lstrip("/")

def parmak_satiri(parmak, oge):
    parmak.update(json.dumps(oge, ensure_ascii=False, separators=(",", ":")).encode("utf-8"))
    parmak.update(b"\n")

beklenen_paket_sha = man.get("training_package_sha256", "")
assert len(beklenen_paket_sha) == 64 and all(c in "0123456789abcdef" for c in beklenen_paket_sha)
parmak = hashlib.sha256()
parmak_satiri(parmak, ["metadata", man["training_package_fingerprint_version"],
    man["package_schema_version"], man["source_dataset_fingerprint"],
    man["archive_root"], man["jpeg_quality"], man["max_edge"], man["classes"]])
dogrulanmis_gorseller = {}
gorulen_girdiler = set()
for split, ornekler in man["splits"].items():
    for i, o in enumerate(ornekler):
        gorsel_yolu = veri_altinda(o["image"], f"{split}[{i}].image")
        etiket_yolu = veri_altinda(o["label"], f"{split}[{i}].label")
        assert o["label"] == doclayout_etiket_yolu(o["image"]), f"{split}[{i}] label eşleşmiyor"
        assert o["image"] not in gorulen_girdiler and o["label"] not in gorulen_girdiler
        gorulen_girdiler.update((o["image"], o["label"]))
        gorsel_sha = dosya_sha256(gorsel_yolu)
        etiket_sha = dosya_sha256(etiket_yolu)
        kaynak_gorsel_sha = o.get("source_encoded_image_sha256", "")
        kaynak_etiket_sha = o.get("source_encoded_label_sha256", "")
        assert len(kaynak_gorsel_sha) == 64 and all(c in "0123456789abcdef" for c in kaynak_gorsel_sha)
        assert len(kaynak_etiket_sha) == 64 and all(c in "0123456789abcdef" for c in kaynak_etiket_sha)
        assert gorsel_sha == o["image_sha256"]
        assert etiket_sha == o["label_sha256"]
        parmak_satiri(parmak, [split, o["image"], o["label"], gorsel_sha, etiket_sha,
            kaynak_gorsel_sha, kaynak_etiket_sha])
        dogrulanmis_gorseller[(split, i)] = gorsel_yolu
parmak_satiri(parmak, ["base_model", base_rel, RESMI_BASE_SHA256, RESMI_BASE_BOYUT])
assert parmak.hexdigest() == beklenen_paket_sha, "eğitim paketi fingerprint uyuşmuyor"
print("Tüm türetilmiş eğitim baytları ve paket fingerprint'i doğrulandı.")

# Yalnız train+validation eğitime girer; calibration/holdout KASITLI dışarıda
# (Mac'teki değerlendirme + kalite kapısı onları kullanır).
liste = {}
for split, dosya in (("train", "train.txt"), ("validation", "val.txt")):
    yol = "/content/" + dosya
    with open(yol, "w") as f:
        for i, _o in enumerate(man["splits"][split]):
            f.write(dogrulanmis_gorseller[(split, i)] + "\n")
    liste[split] = yol
    print(f"{split}: {len(man['splits'][split])} sayfa")

DATA_YAML = "/content/question_layout_v4_palme.yaml"
with open(DATA_YAML, "w") as f:
    f.write(f'path: "{VERI}"\n')
    f.write(f'train: "{liste["train"]}"\n')
    f.write(f'val: "{liste["validation"]}"\n')
    f.write("names:\n")
    for i, s in enumerate(SINIFLAR):
        f.write(f'  {i}: "{s}"\n')
print("YAML:", DATA_YAML)


In [ ]:
from contextlib import contextmanager
from doclayout_yolo import YOLOv10

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
BATCH = 8 if vram > 30 else (6 if vram > 20 else 4)
WORKERS = 2
print(f"GPU: {torch.cuda.get_device_name(0)} ({vram:.0f} GB) "
      f"→ batch={BATCH}, workers={WORKERS}")

assert RESUME is False, "Güvenlik: doğrulanmamış pickle checkpoint resume kapalı"
RUN_START_NS = __import__("time").time_ns()
best_beklenen = os.path.join(CIKTI_DIZIN, "model", "weights", "best.pt")
assert not os.path.exists(best_beklenen), "Eski best.pt var; yeni RUN_ID kullanın"

@contextmanager
def dogrulanmis_checkpoint_yukleyici(yol, beklenen_sha):
    asil_load = torch.load
    izinli = os.path.realpath(yol)
    def sinirli_load(*args, **kwargs):
        assert args, "torch.load yolu eksik"
        gelen = os.path.realpath(os.fspath(args[0]))
        assert gelen == izinli, f"doğrulanmamış checkpoint reddedildi: {gelen}"
        assert dosya_sha256(gelen) == beklenen_sha, "checkpoint SHA değişti"
        kwargs["weights_only"] = False
        return asil_load(*args, **kwargs)
    torch.load = sinirli_load
    try:
        yield
    finally:
        torch.load = asil_load

with dogrulanmis_checkpoint_yukleyici(BASLANGIC, base_bilgi["sha256"]):
    model = YOLOv10(BASLANGIC)
TRAIN_COMPLETED = False
model.train(
        data=DATA_YAML, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
        device=0, workers=WORKERS, patience=PATIENCE,
        project=CIKTI_DIZIN, name="model", exist_ok=True, resume=False,
        amp=True,                              # CUDA'da güvenli (yereldeki
                                               # amp=False yalnız MPS içindi)
        fliplr=0.0, flipud=0.0, degrees=0.0,   # metin belgesi: ayna/döndürme yok
        mosaic=0.5, close_mosaic=10,
        hsv_h=0.01, hsv_s=0.3, hsv_v=0.2,
        translate=0.05, scale=0.2,
        seed=42, deterministic=True, verbose=True)
TRAIN_COMPLETED = True


In [ ]:
import time

best = best_beklenen
taze = os.path.isfile(best) and os.stat(best).st_mtime_ns >= RUN_START_NS
assert TRAIN_COMPLETED is True, "Eğitim normal biçimde tamamlanmadı"
assert taze, "Bu koşuda üretilmiş taze best.pt yok"
sonuc = {"ok": True, "run_id": RUN_ID,
         "best_path": best if os.path.isfile(best) else None,
         "best_sha256": dosya_sha256(best) if os.path.isfile(best) else None,
         "best_size": os.path.getsize(best) if os.path.isfile(best) else None,
         "training_package_sha256": man["training_package_sha256"],
         "source_dataset_fingerprint": man["source_dataset_fingerprint"],
         "archive_sha256": ZIP_SHA256,
         "base_model_sha256": base_bilgi["sha256"],
         "config": {"epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH,
                    "workers": WORKERS, "patience": PATIENCE},
         "started_at_ns": RUN_START_NS,
         "finished_at_ns": time.time_ns(), "resume": False}
with open(os.path.join(CIKTI_DIZIN, "sonuc.json"), "w") as f:
    json.dump(sonuc, f, ensure_ascii=False, indent=1)
print(json.dumps(sonuc, ensure_ascii=False, indent=1))
print("\n✓ best.pt Drive'da:",
      best.replace("/content/drive/MyDrive/", "MyDrive/"))


## Eğitim sonrası (Mac'te)

1. Drive'dan `soru-kesici/ql_v4_runs/palme-all-v4-20260731-r1/model/weights/best.pt` ve `sonuc.json` dosyalarını indirin
   (ör. `~/Downloads/best.pt`).
2. Dört sınıflı holdout + eğitim/paket SHA bağlı, kalite kapılı yayın
   (geçemezse aktif sürüm işaretçisi değişmez):

```bash
cd ~/Developer/soru-kesici
.venv/bin/python soru_model_egitim.py \
  "$HOME/Library/Application Support/soru-kesici/egitim/ql_v4_palme_all/manifest.json" \
  --yayinla ~/Downloads/best.pt --sonuc ~/Downloads/sonuc.json \
  --paket-manifest ~/Desktop/ql_v4_palme_all_colab.zip.manifest.json \
  --surum v4.0.0-palme-all
```

3. Yayın sonrası regresyon kontrolü: `.venv/bin/python kalite_kosusu.py --ikili`
